# Functional Boundary Oracle 001

Hosted accelerator launcher only. Scientific logic and gates live in the repository protocol and runners. Each formal seed is published immediately before the next seed begins.

In [ ]:
from pathlib import Path
import os, subprocess, sys

ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/functional-boundary-oracle-001'
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
subprocess.run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], cwd=ROOT, check=True)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip())

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.0.0', 'huggingface_hub==1.11.0', 'safetensors==0.7.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm,dev]'], cwd=ROOT, check=True)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
    try:
        os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    except Exception:
        pass
except Exception as exc:
    raise RuntimeError('Kaggle Secret GITHUB_TOKEN is required') from exc

subprocess.run([
    sys.executable, 'scripts/research/functional_boundary_oracle_001/publish.py',
    '--branch', BRANCH, '--preflight-only'
], cwd=ROOT, check=True)

In [ ]:
import json, torch, transformers, huggingface_hub, safetensors
assert torch.cuda.is_available(), 'CUDA is required for the formal run'
print({
    'gpu': torch.cuda.get_device_name(0),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'huggingface_hub': huggingface_hub.__version__,
    'safetensors': safetensors.__version__,
})
protocol = json.loads((ROOT / 'research/validations/functional-boundary-oracle-001/protocol.json').read_text())
formal_seeds = protocol['formal_seeds']
formal_seeds

In [ ]:
subprocess.run([sys.executable, 'scripts/research/functional_boundary_oracle_001/kaggle_preflight.py', '--minimum-free-mb', '12000'], cwd=ROOT, check=True)

In [ ]:
def run_logged(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            handle.write(line)
            handle.flush()
        returncode = process.wait()
    if returncode != 0:
        subprocess.run(['nvidia-smi'], check=False)
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-120:]
        print('\n=== Last child log lines ===')
        print('\n'.join(tail))
        raise RuntimeError(f'formal child failed with exit code {returncode}; full log: {log_path}')

for seed in formal_seeds:
    durable = ROOT / f'artifacts/experiments/functional-boundary-oracle-001/seed-{seed}/result.json'
    if durable.is_file():
        print(f'[seed={seed}] already published; skipping')
        continue
    subprocess.run([sys.executable, 'scripts/research/functional_boundary_oracle_001/kaggle_preflight.py', '--minimum-free-mb', '12000'], cwd=ROOT, check=True)
    print(f'[seed={seed}] running formal GPU experiment')
    run_logged([
        sys.executable, 'scripts/research/functional_boundary_oracle_001/run_formal_seed.py',
        '--seed', str(seed), '--device', 'cuda:0'
    ], ROOT / f'results/functional-boundary-oracle-001-launcher/seed-{seed}.log')
    print(f'[seed={seed}] publishing before continuing')
    subprocess.run([
        sys.executable, 'scripts/research/functional_boundary_oracle_001/publish.py',
        '--seed', str(seed), '--branch', BRANCH
    ], cwd=ROOT, check=True)
    decision = json.loads((ROOT / 'artifacts/experiments/functional-boundary-oracle-001/decision.json').read_text())
    print(json.dumps(decision, indent=2, sort_keys=True))

If a Kaggle session stops, rerun the notebook. Already-published seeds are skipped because the branch itself is the durable checkpoint. A scientific FAIL is still published. The formal runner now infers Granite expert width from the actual packed tensor geometry rather than the model-level intermediate_size field.